# Tutoring Center Utilization & Attendance Analysis
**Question:** When are tutoring sessions best and worst attended, and what drives demand?

## 1. Load the data

In [ ]:
# Imports
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency

# Upload the Excel file (a "Choose Files" button appears in Colab)
from google.colab import files
uploaded = files.upload()

# Read the uploaded file into a DataFrame
filename = list(uploaded.keys())[0]
df = pd.read_excel(filename)   # use pd.read_csv(filename) for a CSV

print(df.shape)   # expect (2035, 10)
df.head()

## 2. Clean the data
Column names and values had stray spaces, a hidden non-breaking character, and inconsistent capitalization.

In [ ]:
# Clean column names: remove hidden non-breaking spaces and extra spaces
df.columns = df.columns.str.replace("\xa0", "", regex=False).str.strip()

# Clean text values
df["Status"] = df["Status"].str.strip()
df["TIME"] = df["TIME"].str.strip()
df["TUTORING SESSION - MODALITY"] = (
    df["TUTORING SESSION - MODALITY"].str.strip().str.title()   # merges "In-person" / "In-Person"
)

# Attendance flag: 1 = attended, 0 = cancelled or no show
df["attended"] = (df["Status"] == "Attended").astype(int)

print(df["Status"].value_counts())
print("Overall attendance:", round(df["attended"].mean() * 100, 1), "%")

## 3. Attendance by time slot
Only slots with 100+ sessions are kept; the rest are rare and have typos.

In [ ]:
by_time = (
    df.groupby("TIME")["attended"]
      .agg(attendance_rate="mean", sessions="count")   # like a pivot table
      .query("sessions >= 100")
      .sort_values("attendance_rate")
)
by_time["attendance_rate"] = (by_time["attendance_rate"] * 100).round(1)
by_time

In [ ]:
# Put slots in chronological order for the chart
order = ["11am - 12pm", "12pm - 1pm", "1pm - 2pm", "2pm - 3pm", "3pm - 4pm",
         "4pm - 5pm", "5pm - 6pm", "6pm - 7pm", "7pm - 8pm"]
chart_data = by_time.loc[order, "attendance_rate"]

# Highlight best (green) and worst (red) slots
colors = ["#2a9d8f" if s == "12pm - 1pm" else "#e76f51" if s == "7pm - 8pm" else "#b0b0b0"
          for s in chart_data.index]

plt.figure(figsize=(10, 5))
bars = plt.bar(chart_data.index, chart_data.values, color=colors)
plt.axhline(79.3, color="black", linestyle="--", linewidth=1, label="Overall average (79.3%)")

# Label each bar with its value
for bar in bars:
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
             f"{bar.get_height():.1f}%", ha="center", fontsize=9)

plt.title("Tutoring Attendance Rate by Time Slot")
plt.ylabel("Attendance rate (%)")
plt.xlabel("Time slot")
plt.ylim(60, 90)   # y-axis starts at 60 so differences are visible
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

## 4. Is the gap statistically significant?
Chi-square test of independence: 12-1pm vs. 7-8pm attendance.

In [ ]:
# Keep only the best and worst slots
two = df[df["TIME"].isin(["12pm - 1pm", "7pm - 8pm"])]

# Counts of not attended (0) vs. attended (1) per slot
table = pd.crosstab(two["TIME"], two["attended"])
print(table)

chi2, p, dof, expected = chi2_contingency(table)
print("p-value:", p)   # below 0.05 = unlikely to be random noise

## 5. Other resume metrics: modality, ASAP share, Math share

In [ ]:
# In-person vs. virtual split
print((df["TUTORING SESSION - MODALITY"].value_counts(normalize=True) * 100).round(1))

# Share of sessions from ASAP students
asap = df["Are you affiliated with any of the programs listed below?"].str.contains("ASAP")
print("ASAP share:", round(asap.mean() * 100, 1), "%")

# Share of sessions in Math (subject codes start with MAT)
math = df["SUBJECT"].str.strip().str.upper().str.startswith("MAT")
print("Math share:", round(math.mean() * 100, 1), "%")

## 6. Sessions by subject

In [ ]:
# Extract the main subject (first word of the free-text SUBJECT column)
subject = df["SUBJECT"].str.strip().str.upper().str.split().str[0]
subject = subject.replace({"MAT": "MATHEMATICS"}).str.title()

# Keep the top 6 subjects and group everything else as "Other"
top = subject.value_counts().head(6)
other = len(df) - top.sum()
counts = pd.concat([top, pd.Series({"Other": other})])
share = (counts / len(df) * 100).round(1)
print(share)

plt.figure(figsize=(9, 5))
bars = plt.barh(share.index[::-1], share.values[::-1], color="black")
for bar in bars:
    plt.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
             f"{bar.get_width():.1f}%", va="center", fontsize=9)
plt.title("Share of Tutoring Sessions by Subject")
plt.xlabel("% of all sessions")
plt.xlim(0, 50)   # leaves room for the longest label
plt.tight_layout()
plt.show()

## Findings & recommendation
- Overall attendance was 79.3%.
- Attendance peaked at 82.9% (12-1pm) and bottomed at 66.4% (7-8pm); the gap is significant (chi-square p < 0.001).
- 4-5pm (73.7%) is also below average.
- Mathematics was 43% of sessions; ASAP students were 49%.
- **Recommendation:** shift tutor capacity from evening slots toward 12-3pm.

*Limitation: this shows association, not cause; data covers one semester.*